Import necessary libraries


In [1]:
import random, os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from itertools import product

I0000 00:00:1775013941.183700  327917 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775013941.265716  327917 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775013943.608884  327917 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Generate config class


In [2]:
class Config:
    def __init__(self):
        # Data params
        self.vocab_size = 10000
        self.max_len = 300
        self.sample_size = 15000

        # Experiment params
        self.models_to_run = ["RNN", "LSTM", "GRU"]

        # Reproducibility
        self.seed = 42
        self.grid = {
            "embedding_dim": [32, 64],
            "hidden_units": [[64], [32, 64]],
            "dropout": [0.0, 0.2],
            "batch_size": [64,128],
            "epochs": [20,30],
            "learning_rate": [0.005,0.01]
        }

    @property
    def num_layers(self):
        return len(self.hidden_units)

Data handeling class


In [3]:
class DataLoader:
    def __init__(self, config):
        self.config = config

    def load_data(self):
        (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=self.config.vocab_size)

        # Sampling control
        if self.config.sample_size:
            x_train = x_train[:self.config.sample_size]
            y_train = y_train[:self.config.sample_size]
            x_test = x_test[:self.config.sample_size]
            y_test = y_test[:self.config.sample_size]

        # Padding
        x_train = pad_sequences(x_train, maxlen=self.config.max_len)
        x_test = pad_sequences(x_test, maxlen=self.config.max_len)

        return x_train, y_train, x_test, y_test

Model builder class


In [4]:
class ModelBuilder:
    def __init__(self, config):
        self.config = config

    def build(self, model_type):
        model = Sequential()

        model.add(Embedding(input_dim=self.config.vocab_size,
                            output_dim=self.config.embedding_dim,
                            input_length=self.config.max_len))

        # Multilayer architecture
        for i, units in enumerate(self.config.hidden_units):

            return_seq = True if i < self.config.num_layers - 1 else False

            if model_type == "RNN":
                model.add(SimpleRNN(units, return_sequences=return_seq))

            elif model_type == "LSTM":
                model.add(LSTM(units, return_sequences=return_seq))

            elif model_type == "GRU":
                model.add(GRU(units, return_sequences=return_seq))

            else:
                raise ValueError("Invalid model type")

        model.add(Dropout(self.config.dropout))
        model.add(Dense(1, activation='sigmoid'))

        model.compile(
            loss='binary_crossentropy',
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config.learning_rate),
            metrics=['accuracy']
        )
        return model

Trainer class


In [5]:
class Trainer:
    def __init__(self, config):
        self.config = config

	# Model training
    def train(self, model, x_train, y_train, x_val, y_val, model_type):
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        history = model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=[early_stop],
            verbose=1
        )

        self.plot_loss(history, model_type)

        return history

	# Loss plot
    def plot_loss(self, history, model_type):
        if not os.path.exists("Results/plots"):
            os.makedirs("Results/plots")

        plt.figure()

        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Validation Loss')

        plt.title(f"{model_type} Loss Curve")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()

        plt.savefig(f"results/plots/{model_type}_layers{self.config.num_layers}.png")
        plt.close()

Evaluator class


In [6]:
class Evaluator:
    def evaluate(self, model, x_test, y_test):
        y_pred_prob = model.predict(x_test)
        y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        return acc, prec, rec, f1

Experiment running pipeline


In [7]:
class ExperimentRunner:
    def __init__(self, config):
        self.config = config
        self.results = []

    def run(self):
        # Reproducibility
        np.random.seed(self.config.seed)
        tf.random.set_seed(self.config.seed)
        random.seed(self.config.seed)

        # Load data
        data_loader = DataLoader(self.config)
        x_train, y_train, x_test, y_test = data_loader.load_data()

        # Validation split
        split = int(0.8 * len(x_train))
        x_val = x_train[split:]
        y_val = y_train[split:]
        x_train = x_train[:split]
        y_train = y_train[:split]

        model_builder = ModelBuilder(self.config)
        trainer = Trainer(self.config)
        evaluator = Evaluator()

        # Grid combinations
        keys = list(self.config.grid.keys())
        values = list(self.config.grid.values())

        for combo in product(*values):

            # 🔥 dynamically update config
            for k, v in zip(keys, combo):
                setattr(self.config, k, v)

            print(f"\n🔥 Running config: {dict(zip(keys, combo))}")

            for model_type in self.config.models_to_run:
                print(f"--- Model: {model_type} ---")

                model = model_builder.build(model_type)

                start = time.time()

                trainer.train(model, x_train, y_train, x_val, y_val, model_type)

                end = time.time()
                train_time = end - start

                acc, prec, rec, f1 = evaluator.evaluate(model, x_test, y_test)

                self.results.append({
                    "model": model_type,
                    "layers": self.config.num_layers,
                    "hidden_units": str(self.config.hidden_units),
                    "embedding_dim": self.config.embedding_dim,
                    "dropout": self.config.dropout,
                    "batch_size": self.config.batch_size,
                    "epochs": self.config.epochs,
                    "accuracy": acc,
                    "precision": prec,
                    "recall": rec,
                    "f1_score": f1,
                    "train_time_sec": train_time
                })

        self.save_results()
        self.show_summary()

    def save_results(self):
        df = pd.DataFrame(self.results)

        if not os.path.exists("Results"):
            os.makedirs("Results")

        file_path = "Results/comparison.csv"

        df.to_csv(
            file_path,
            mode='a',
            index=False,
            header=not os.path.exists(file_path)
        )

        print("\n✅ Results saved at Results/comparison.csv")

    def show_summary(self):
        df = pd.DataFrame(self.results)

        print("\n================= SUMMARY TABLE =================")
        print(df)

        print("\n================= TOP 5 (by F1 Score) =================")
        print(df.sort_values(by="f1_score", ascending=False).head(5))

        print("\n================= BEST CONFIG =================")
        best = df.sort_values(by="f1_score", ascending=False).iloc[0]
        print(best)

In [ ]:
Experiment = ExperimentRunner(Config())
Experiment.run()

/home/user/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)



🔥 Running config: {'embedding_dim': 32, 'hidden_units': [64], 'dropout': 0.0, 'batch_size': 64, 'epochs': 20, 'learning_rate': 0.005}
--- Model: RNN ---


/home/user/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
W0000 00:00:1775013951.352656  327917 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 20s 94ms/step - accuracy: 0.5236 - loss: 0.6975 - val_accuracy: 0.5550 - val_loss: 0.6806
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 18s 98ms/step - accuracy: 0.6470 - loss: 0.6276 - val_accuracy: 0.6280 - val_loss: 0.6399
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 98ms/step - accuracy: 0.7539 - loss: 0.5016 - val_accuracy: 0.7223 - val_loss: 0.5606
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 98ms/step - accuracy: 0.8312 - loss: 0.3891 - val_accuracy: 0.6813 - val_loss: 0.6306
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - accuracy: 0.8489 - loss: 0.3572 - val_accuracy: 0.7013 - val_loss: 0.6502
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 98ms/step - accuracy: 0.8072 - loss: 0.4144 - val_accuracy: 0.6703 - val_loss: 0.6708
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - accuracy: 0.8781 - loss: 0.3106 - val_accuracy: 0.7363 - val_loss: 0.6935
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 99ms/step - accuracy: 0.9032 - loss: 0.2550 - 

/home/user/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


...

94/94 ━━━━━━━━━━━━━━━━━━━━ 37s 365ms/step - accuracy: 0.7137 - loss: 0.5378 - val_accuracy: 0.8477 - val_loss: 0.3541
Epoch 2/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 30s 320ms/step - accuracy: 0.9013 - loss: 0.2464 - val_accuracy: 0.8713 - val_loss: 0.3485
Epoch 3/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 30s 321ms/step - accuracy: 0.9601 - loss: 0.1161 - val_accuracy: 0.8520 - val_loss: 0.4170
Epoch 4/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 30s 318ms/step - accuracy: 0.9694 - loss: 0.0828 - val_accuracy: 0.8337 - val_loss: 0.4648
Epoch 5/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 30s 324ms/step - accuracy: 0.9822 - loss: 0.0487 - val_accuracy: 0.8517 - val_loss: 0.5595
Epoch 6/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 30s 322ms/step - accuracy: 0.9952 - loss: 0.0157 - val_accuracy: 0.8610 - val_loss: 0.6786
Epoch 7/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 30s 323ms/step - accuracy: 0.9933 - loss: 0.0166 - val_accuracy: 0.8530 - val_loss: 0.7432
469/469 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step

🔥 Running config: {'embedding_dim': 32, 'hidden_units': [64], 'd

/home/user/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 103ms/step - accuracy: 0.5430 - loss: 0.6954 - val_accuracy: 0.5757 - val_loss: 0.6740
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 102ms/step - accuracy: 0.6757 - loss: 0.5945 - val_accuracy: 0.7053 - val_loss: 0.5813
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 99ms/step - accuracy: 0.7172 - loss: 0.5538 - val_accuracy: 0.6370 - val_loss: 0.6424
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 100ms/step - accuracy: 0.7780 - loss: 0.4811 - val_accuracy: 0.6727 - val_loss: 0.6521
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 100ms/step - accuracy: 0.8017 - loss: 0.4520 - val_accuracy: 0.6870 - val_loss: 0.6709
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 101ms/step - accuracy: 0.8347 - loss: 0.3953 - val_accuracy: 0.6797 - val_loss: 0.6864
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 19s 100ms/step - accuracy: 0.8612 - loss: 0.3510 - val_accuracy: 0.6760 - val_loss: 0.7905
469/469 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step
--- Model: LSTM ---
Epoch 1/20


/home/user/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


 50/188 ━━━━━━━━━━━━━━━━━━━━ 25s 182ms/step - accuracy: 0.5405 - loss: 0.6917

In [ ]:
print("Done")